# 02 · Kết quả

Đọc `results/runs_log.csv` (qua `aggregate.py`) và vẽ hình cho báo cáo. Notebook này không huấn luyện gì nên không cần GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/drybean-1d-resnet

## 1. Kiểm tra log + bảng tổng hợp

Nếu in `Log CHƯA đạt` thì dừng lại và sửa log (chạy nốt run thiếu) trước khi đọc bảng.

In [ ]:
!python aggregate.py

In [ ]:
import pandas as pd, numpy as np, os, matplotlib.pyplot as plt
from IPython.display import display, Image
S = 'results/summary'
summ = pd.read_csv(f'{S}/summary.csv')
cols = ['group','model','n_params','test_acc','test_f1_weighted','test_f1_macro','epochs_ran','best_epoch','paper_acc','gap_vs_paper']
display(summ[cols])
display(pd.read_csv(f'{S}/paired_tests.csv'))
display(pd.read_csv(f'{S}/per_class_recall.csv'))

## 2. Accuracy theo group và model (thanh lỗi = std qua 5 split; dấu × = paper)

In [ ]:
PAPER = {'resnet1': 95.70, 'resnet2': 93.79, 'resnet3': 93.71}
groups = list(dict.fromkeys(summ['group']))
models = ['resnet1', 'resnet2', 'resnet3']
w = 0.8 / len(groups)
fig, ax = plt.subplots(figsize=(8, 4))
for i, g in enumerate(groups):
    d = summ[summ.group == g].set_index('model').loc[models]
    ax.bar(np.arange(3) + i * w, d['test_acc_mean'], w, yerr=d['test_acc_std'], capsize=3, label=g)
ax.scatter(np.arange(3) + w * (len(groups) - 1) / 2, [PAPER[m] for m in models], marker='x', s=80, c='k', zorder=5, label='paper')
ax.set_xticks(np.arange(3) + w * (len(groups) - 1) / 2, models)
ax.set_ylim(90, 97); ax.set_ylabel('test accuracy (%)'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{S}/acc_by_group.png', dpi=120); plt.show()

## 3. Learning curve (baseline, split 0)

In [ ]:
log = pd.read_csv('results/runs_log.csv')
log = log[log.status == 'completed']
for m in models:
    r = log[(log.group == 'baseline') & (log.model == m) & (log.split_id == 0)].iloc[0]
    print(r.run_id, '| epochs_ran', r.epochs_ran, '| best_epoch_val_loss', r.best_epoch_val_loss, '| best_epoch_val_acc', r.best_epoch_val_acc)
    display(Image(os.path.join(os.path.dirname(r.ckpt_path), 'curve.png')))

## 4. Confusion matrix cộng dồn 5 split (baseline), chuẩn hoá theo hàng = recall

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
for ax, m in zip(axes, models):
    cm = pd.read_csv(f'{S}/cm_baseline_{m}.csv', index_col=0)
    p = cm.div(cm.sum(1), axis=0) * 100
    ax.imshow(p.values, cmap='Blues', vmin=0, vmax=100)
    for i in range(7):
        for j in range(7):
            ax.text(j, i, f'{p.values[i, j]:.1f}', ha='center', va='center', fontsize=7,
                    color='white' if p.values[i, j] > 60 else 'black')
    ax.set_xticks(range(7), cm.columns, rotation=60, fontsize=7); ax.set_yticks(range(7), cm.index, fontsize=7)
    ax.set_title(m); ax.set_xlabel('dự đoán'); ax.set_ylabel('thật')
plt.tight_layout(); plt.savefig(f'{S}/cm_baseline.png', dpi=120); plt.show()

## 5. noES: model tốt nhất (checkpoint) so với model ở epoch cuối

In [ ]:
d = log[log.group == 'ablation_noES']
if len(d):
    display(d.groupby('model')[['test_acc', 'test_acc_last', 'best_epoch_val_loss']].agg(['mean', 'std']).round(2))
else:
    print('Chưa chạy ablation_noES')